# AIM-Flow Kaggle Demo

This notebook clones the GitHub repo, installs dependencies, authenticates with Hugging Face, loads SD3 Medium, and compares base SD3 vs AIM-Flow.

In [ ]:
# User variables
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Huggingface")

GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/aim-flow.git"
HF_TOKEN = secret_value_0
PROMPT_KEY = "cyborg_dogs"
OUTPUT_DIR = "/kaggle/working/aim_flow_outputs/cyborg_dogs"

In [ ]:
# Clone repo
!git clone {GITHUB_REPO_URL} /kaggle/working/aim-flow
%cd /kaggle/working/aim-flow

In [ ]:
# Install dependencies
# Kaggle may preinstall a newer PyTorch build that drops Tesla P100 / sm_60 support.
# Install PyTorch first from the CUDA 11.8 index, then install the lighter AIM-Flow deps.
!pip uninstall -y -q torch torchvision torchaudio
!pip install -q --no-cache-dir --force-reinstall torch==2.4.1+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -q --no-cache-dir -r requirements-kaggle.txt

In [ ]:
# Hugging Face login/token handling
import os

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        os.environ["HF_TOKEN"] = user_secrets.get_secret("Huggingface")
    except Exception:
        print("No HF token found. Make sure you have accepted SD3 Medium license and added a Kaggle secret named Huggingface.")

In [ ]:
# Check GPU and package versions
import torch
import diffusers

print("torch", torch.__version__, "cuda", torch.version.cuda)
print("diffusers", diffusers.__version__)
print("arch list", torch.cuda.get_arch_list() if torch.cuda.is_available() else [])
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    capability = torch.cuda.get_device_capability(0)
    print("compute capability", capability)
    arch = f"sm_{capability[0]}{capability[1]}"
    if arch not in torch.cuda.get_arch_list():
        raise RuntimeError(f"Installed PyTorch does not support {arch}. Rerun the install cell and restart the Kaggle session if needed.")
    print(torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")
    x = torch.ones(1, device="cuda")
    print("cuda smoke test", x.item())

In [ ]:
# Run the comparison
!python scripts/run_compare.py \
  --config configs/sd3_medium_kaggle.yaml \
  --prompts configs/sample_prompts.yaml \
  --prompt-key {PROMPT_KEY} \
  --output-dir {OUTPUT_DIR} \
  --modes base anchor naive_v1 aim_v2

In [ ]:
# Display grid
from PIL import Image
import matplotlib.pyplot as plt

grid_path = f"{OUTPUT_DIR}/comparison_grid.png"
img = Image.open(grid_path)
plt.figure(figsize=(16, 8))
plt.imshow(img)
plt.axis("off")

In [ ]:
# Show AIM-Flow v2 metadata summary
import json

metadata_path = f"{OUTPUT_DIR}/metadata_aim_v2.json"
with open(metadata_path) as f:
    data = json.load(f)
summary = {
    "method": data.get("method"),
    "primitive_conditioning": data.get("primitive_conditioning"),
    "primitive_original_texts": data.get("primitive_original_texts"),
    "primitive_condition_texts": data.get("primitive_condition_texts"),
    "first_step_debug": (data.get("debug_steps") or [{}])[0],
}
print(json.dumps(summary, indent=2)[:4000])

In [ ]:
# Troubleshooting run: lower memory / faster debug settings
# !python scripts/run_compare.py \
#   --config configs/sd3_medium_kaggle.yaml \
#   --prompts configs/sample_prompts.yaml \
#   --prompt-key cyborg_dogs \
#   --output-dir /kaggle/working/aim_flow_outputs/cyborg_dogs_debug \
#   --modes base anchor naive_v1 aim_v2 \
#   --num-inference-steps 16 \
#   --height 384 --width 384 \
#   --ltp-mode velocity
# You can also reduce aim_flow.max_primitives in configs/sd3_medium_kaggle.yaml or keep CPU offload enabled.

## Notes

- If you see a PyTorch warning that P100 / `sm_60` is unsupported, rerun the install cell. It force-reinstalls `torch==2.4.1+cu118`.
- The Kaggle config disables SD3's T5 text encoder (`text_encoder_3`) to reduce memory pressure.
- If out of memory, reduce steps to 16, use 384x384, set `--ltp-mode velocity`, reduce `max_primitives`, or keep CPU offload enabled.
- If model access fails, accept the model license on Hugging Face and provide HF_TOKEN.
- This prototype avoids VQA/reward models and only compares images qualitatively.